# Aurora temperature verification

This notebook uses the canonical intermediate-reader schema for Aurora. It makes global RMSE maps for the selected leads, then area-weighted RMSE curves for East Africa. `Hot days` means cases where the **observed** ERA5 q95 hot-day event occurred.

The reader and every calculation below remain lazy until the per-lead `compute()` calls.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import cartopy.crs as ccrs
from dask.diagnostics import ProgressBar

import heatextremes.verification.case_cache_reader as reader

In [ ]:
MODEL_NAME = "aurora_e2s"
SELECTED_LEADS = [0, 5, 10]
JJAS_MONTHS = [6, 7, 8, 9]

# Default region from configs/verification/regions.yaml. Change these bounds
# later to evaluate another region.
REGION_NAME = "East Africa"
REGION = {"latitude": (-15.0, 20.0), "longitude": (25.0, 55.0)}

In [ ]:
# Modern models discover their available month/lead cache stores from the
# inventory manifest. The reader logs any absent lead stores and fills them
# with NaNs, so the following metrics safely skip unavailable cases.
ds = reader.open_model_intermediates(
    MODEL_NAME,
    forecast_days=SELECTED_LEADS,
)

jjas = ds["initialization"].dt.month.isin(JJAS_MONTHS)
ds = ds.sel(initialization=jjas)
ds

In [ ]:
def coordinate_slice(coordinate, lower, upper):
    """Return a slice that works for ascending or descending coordinates."""
    return slice(lower, upper) if coordinate.values[0] < coordinate.values[-1] else slice(upper, lower)


def select_region(values, region):
    """Subset a non-seam-crossing latitude/longitude box."""
    latitude_min, latitude_max = region["latitude"]
    longitude_min, longitude_max = region["longitude"]
    return values.sel(
        latitude=coordinate_slice(values.latitude, latitude_min, latitude_max),
        longitude=coordinate_slice(values.longitude, longitude_min, longitude_max),
    )


def weighted_rmse(error):
    """Area-weighted RMSE over initialization, latitude, and longitude."""
    weights = xr.DataArray(
        np.cos(np.deg2rad(error.latitude)),
        dims=("latitude",),
        coords={"latitude": error.latitude},
    )
    valid = error.notnull()
    numerator = ((error**2) * weights).sum(
        ("initialization", "latitude", "longitude"), skipna=True
    )
    denominator = weights.where(valid).sum(
        ("initialization", "latitude", "longitude"), skipna=True
    )
    return np.sqrt(numerator / denominator)


def error_cases(dataset, hot_only=False):
    """Return temperature errors for all valid cases or observed-hot valid cases."""
    error = (
        dataset["forecast_temperature"] - dataset["observation_temperature"]
    ).where(dataset["temperature_case_valid"])
    if not hot_only:
        return error

    observed_hot = dataset["observed_event"].sel(event="hot_day_q95") > 0.5
    event_valid = dataset["event_case_valid"].sel(event="hot_day_q95").fillna(False).astype(bool)
    return error.where(observed_hot & event_valid)


def lead_statistics(dataset, forecast_day, region):
    """Compute maps and regional RMSE for one lead in one bounded Dask graph."""
    lead = dataset.sel(forecast_day=forecast_day)
    all_error = error_cases(lead)
    hot_error = error_cases(lead, hot_only=True)
    regional_all_error = select_region(all_error, region)
    regional_hot_error = select_region(hot_error, region)

    result = xr.Dataset(
        {
            "all_rmse_map": np.sqrt((all_error**2).mean("initialization", skipna=True)),
            "hot_rmse_map": np.sqrt((hot_error**2).mean("initialization", skipna=True)),
            "all_regional_rmse": weighted_rmse(regional_all_error),
            "hot_regional_rmse": weighted_rmse(regional_hot_error),
            "all_regional_cases": regional_all_error.notnull().sum(),
            "hot_regional_cases": regional_hot_error.notnull().sum(),
        }
    )
    with ProgressBar():
        return result.compute()


def plot_rmse_map(field, *, title, vmax, region):
    fig, axis = plt.subplots(
        figsize=(12, 5),
        subplot_kw={"projection": ccrs.PlateCarree()},
        constrained_layout=True,
    )
    field.plot.pcolormesh(
        ax=axis,
        transform=ccrs.PlateCarree(),
        cmap="magma_r",
        vmin=0,
        vmax=vmax,
        cbar_kwargs={"label": "RMSE (K)"},
    )
    axis.coastlines(linewidth=0.6)
    axis.set_global()
    axis.add_patch(
        Rectangle(
            (region["longitude"][0], region["latitude"][0]),
            region["longitude"][1] - region["longitude"][0],
            region["latitude"][1] - region["latitude"][0],
            fill=False,
            edgecolor="deepskyblue",
            linewidth=1.5,
            transform=ccrs.PlateCarree(),
        )
    )
    axis.set_title(title)
    plt.show()

In [ ]:
results_by_lead = {}
regional_rows = []

for forecast_day in SELECTED_LEADS:
    print(f"Computing Aurora forecast day {forecast_day}…", flush=True)
    result = lead_statistics(ds, forecast_day, REGION)
    results_by_lead[forecast_day] = result
    regional_rows.append(
        {
            "forecast_day": forecast_day,
            "all_rmse": float(result["all_regional_rmse"]),
            "hot_rmse": float(result["hot_regional_rmse"]),
            "all_cases": int(result["all_regional_cases"]),
            "hot_cases": int(result["hot_regional_cases"]),
        }
    )

regional_rmse = pd.DataFrame(regional_rows).sort_values("forecast_day")
regional_rmse

In [ ]:
all_vmax = max(float(result["all_rmse_map"].quantile(0.98, skipna=True)) for result in results_by_lead.values())
hot_vmax = max(float(result["hot_rmse_map"].quantile(0.98, skipna=True)) for result in results_by_lead.values())

for forecast_day, result in results_by_lead.items():
    plot_rmse_map(
        result["all_rmse_map"],
        title=f"Aurora RMSE — all valid cases — forecast day {forecast_day}",
        vmax=all_vmax,
        region=REGION,
    )
    plot_rmse_map(
        result["hot_rmse_map"],
        title=f"Aurora RMSE — observed hot days only — forecast day {forecast_day}",
        vmax=hot_vmax,
        region=REGION,
    )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharex=True, constrained_layout=True)

axes[0].plot(regional_rmse["forecast_day"], regional_rmse["all_rmse"], marker="o")
axes[0].set(
    title=f"Aurora RMSE — all valid cases — {REGION_NAME}",
    xlabel="Forecast day",
    ylabel="Area-weighted RMSE (K)",
)

axes[1].plot(regional_rmse["forecast_day"], regional_rmse["hot_rmse"], marker="o", color="firebrick")
axes[1].set(
    title=f"Aurora RMSE — observed hot days only — {REGION_NAME}",
    xlabel="Forecast day",
    ylabel="Area-weighted RMSE (K)",
)

for axis in axes:
    axis.grid(alpha=0.3)
    axis.set_xticks(SELECTED_LEADS)

plt.show()
regional_rmse